# BoltzDesign1 → ProteinMPNN: RBX1 RING Core Binder Design

**Target**: RBX1 RING core, residues 32–83 (52 AA, chain B)  
**Why truncated**: Full 108 AA RBX1 has disordered N-term (1–31) and C-term (84–108) tails that attract false binders. This uses only the structured zinc-binding core.

**Pipeline**:
1. BoltzDesign1 — gradient-based backbone generation conditioned on RING hotspots
2. ProteinMPNN — sequence redesign of the binder (chain A), RBX1 fixed (chain B)
3. Boltz-2 — rescore all sequences, filter by ipTM ≥ 0.70

**Hotspot residues** (1-based in the 52 AA fragment, corresponding to original positions 35,37,43,45,46,54,55,57,59 in RING domain):  
`4, 6, 12, 14, 15, 23, 24, 26, 28, 52`

In [ ]:
#@title 🛠️ Setup — BoltzDesign1 + ProteinMPNN (~5 minutes)
%%time
import os

if not os.path.isdir('BoltzDesign1'):
    !git clone -q https://github.com/yehlincho/BoltzDesign1.git
    !cd /content/BoltzDesign1/boltz && pip install -q git+https://github.com/prody/ProDy.git .
    !pip install -q pypdb py3Dmol logmd
    !cd /content/BoltzDesign1/LigandMPNN && bash get_model_params.sh './model_params'
    exit()  # kernel restarts here — re-run all cells after restart

if not os.path.isdir('ProteinMPNN'):
    !git clone -q https://github.com/dauparas/ProteinMPNN.git

print('Setup complete')

In [ ]:
#@title 🛠️ Load BoltzDesign1 model
from boltz.main import download
from pathlib import Path
download(Path('.'))

import os, sys, shutil
sys.path.append('./BoltzDesign1/boltzdesign')
from boltzdesign_utils import *
from input_utils import *
from utils import *
import pandas as pd, yaml

ccd_path = 'ccd.pkl'
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
predict_args = {
    'recycling_steps': 0,
    'sampling_steps': 200,
    'diffusion_samples': 1,
    'write_confidence_summary': True,
    'write_full_pae': False,
    'write_full_pde': False,
}
boltz_model = get_boltz_model('boltz1_conf.ckpt', predict_args, device)
boltz_model.train()
print('BoltzDesign1 model loaded')

In [ ]:
#@title 📂 Upload target PDB
# Upload rbx1_ring_core.pdb  (residues 32-83, chain B, 52 AA)
# This is the structured RING core — no disordered tails.
from google.colab import files
print('Upload rbx1_ring_core.pdb')
uploaded = files.upload()
TARGET_PDB = list(uploaded.keys())[0]
print(f'Loaded: {TARGET_PDB}')

# Verify
residues = set()
chain_seen = set()
with open(TARGET_PDB) as f:
    for line in f:
        if line.startswith('ATOM'):
            residues.add(int(line[22:26]))
            chain_seen.add(line[21])
print(f'Chains: {sorted(chain_seen)}  |  Residues: {min(residues)}-{max(residues)} (n={len(residues)})')

In [ ]:
#@title 📄 Generate YAML for BoltzDesign1
import warnings, logging
warnings.simplefilter('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s - %(message)s')

TARGET_NAME   = 'rbx1_ring_core'
TARGET_CHAIN  = 'B'        # chain in the PDB
BINDER_CHAIN  = 'A'

# Hotspot residues for pocket conditioning
# 1-based positions in the 52 AA fragment (32-83 of full RBX1)
# Corresponds to RING H2 loop / zinc-coordinating surface
CONTACT_RESIDUES = '4,6,12,14,15,23,24,26,28,52'

config_obj = Config(main_dir=f'/content/inputs/{TARGET_NAME}_binder')
config_obj.setup_directories()

yaml_dict, yaml_dir = generate_yaml_config(
    input_type='pdb',
    pdb_path=TARGET_PDB,
    target_type='protein',
    target_name=TARGET_NAME,
    pdb_target_ids=TARGET_CHAIN,
    target_mols='',
    custom_target_input='',
    custom_target_ids='',
    binder_id=BINDER_CHAIN,
    use_msa=False,
    contact_residues=CONTACT_RESIDUES,
    modifications='',
    modifications_positions='',
    modification_target='',
    constraint_target=TARGET_CHAIN,
    config_obj=config_obj
)

# Load PPI config
config_path = '/content/BoltzDesign1/boltzdesign/configs/default_ppi_config.yaml'
with open(config_path) as f:
    config = yaml.safe_load(f)

print(f'YAML saved: {yaml_dir}')
print(f'Pocket conditioning: {bool(CONTACT_RESIDUES)} — hotspots {CONTACT_RESIDUES}')

In [ ]:
#@title 🧬 Design settings

DESIGN_SAMPLES = 20     #@param {type:"integer"}  — number of backbones to generate
LENGTH_MIN     = 65     #@param {type:"integer"}  — min binder length (AA)
LENGTH_MAX     = 95     #@param {type:"integer"}  — max binder length (AA)
OPTIMIZER      = 'SGD'  #@param ["SGD", "AdamW"]

config['binder_chain']               = BINDER_CHAIN
config['non_protein_target']         = False
config['msa_max_seqs']               = 4096
config['length_min']                 = LENGTH_MIN
config['length_max']                 = LENGTH_MAX
config['optimizer_type']             = OPTIMIZER
config['pocket_conditioning']        = True
config['optimize_contact_per_binder_pos'] = True   # correct for PPI
config['mask_ligand']                = False        # no ligand mask for protein target

# Iteration schedule
config['pre_iteration']    = 30
config['soft_iteration']   = 80
config['temp_iteration']   = 45
config['hard_iteration']   = 5
config['semi_greedy_steps']= 2

# Loss weights (PPI)
loss_scales = {
    'con_loss':   1.0,    # intra-binder contacts
    'i_con_loss': 1.0,    # binder-RBX1 interface contacts
    'plddt_loss': 0.1,
    'pae_loss':   0.4,
    'i_pae_loss': 0.1,
    'rg_loss':    0.0,
}

print(f'Binder length: {LENGTH_MIN}–{LENGTH_MAX} AA')
print(f'Designs to generate: {DESIGN_SAMPLES}')
print(f'Pocket conditioning: ON  |  hotspots: {CONTACT_RESIDUES}')

In [ ]:
#@title 🚀 Run BoltzDesign1
import warnings
warnings.simplefilter('ignore', DeprecationWarning)

MAIN_DIR     = 'outputs'
VERSION_NAME = f'protein_{TARGET_NAME}'
os.makedirs(MAIN_DIR, exist_ok=True)

boltz_path = shutil.which('boltz')

run_boltz_design(
    boltz_path=boltz_path,
    main_dir=MAIN_DIR,
    yaml_dir=os.path.dirname(yaml_dir),
    boltz_model=boltz_model,
    ccd_path=ccd_path,
    design_samples=DESIGN_SAMPLES,
    version_name=VERSION_NAME,
    config=config,
    loss_scales=loss_scales,
    num_workers=0,
    show_animation=True,
    save_trajectory=False,
    redo_boltz_predict=False,
)

print('\nBoltzDesign1 complete')

In [ ]:
#@title 📊 BoltzDesign1 results
results_csv = os.path.join(MAIN_DIR, VERSION_NAME, 'results_final', 'rmsd_results.csv')
df = pd.read_csv(results_csv)
print(f'Generated {len(df)} backbones')
print(df[['target','length','iptm','complex_plddt','apo_holo_rmsd']].sort_values('iptm', ascending=False).to_string())

# Filter backbones worth redesigning (iptm > 0.5)
good = df[df['iptm'] >= 0.50]
print(f'\nBackbones with ipTM >= 0.50: {len(good)}')

In [ ]:
#@title 🔄 Convert BoltzDesign1 CIF outputs → PDB for ProteinMPNN
import glob, re

results_dir = os.path.join(MAIN_DIR, VERSION_NAME, 'results_final')
pdb_dir     = os.path.join(MAIN_DIR, VERSION_NAME, 'pdb')
os.makedirs(pdb_dir, exist_ok=True)

AA3 = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
       'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
       'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

def cif_to_pdb(cif_path, pdb_path):
    """Convert mmCIF to minimal PDB, keeping both chains (A=binder, B=target)."""
    cols = {}; lines_out = []; in_loop = False; serial = 1
    with open(cif_path) as f:
        for line in f:
            line = line.rstrip()
            if line.startswith('_atom_site.'):
                cols[line.split('.')[-1].strip()] = len(cols)
                in_loop = True; continue
            if not in_loop or not line.startswith('ATOM'): continue
            parts = line.split()
            try:
                atom    = parts[cols['label_atom_id']]
                chain   = parts[cols['label_asym_id']]
                resnum  = int(parts[cols['label_seq_id']])
                resname = parts[cols['label_comp_id']]
                x = float(parts[cols['Cartn_x']])
                y = float(parts[cols['Cartn_y']])
                z = float(parts[cols['Cartn_z']])
            except: continue
            if resname not in AA3: continue
            atom_fmt = f' {atom:<3s}' if len(atom) < 4 else atom
            lines_out.append(
                f'ATOM  {serial:5d} {atom_fmt} {resname} {chain}{resnum:4d}    '
                f'{x:8.3f}{y:8.3f}{z:8.3f}  1.00  0.00           {atom[0]}\n'
            )
            serial += 1
    with open(pdb_path, 'w') as f:
        f.writelines(lines_out)
        f.write('END\n')

# Convert all CIFs in results_final
cif_files = sorted(glob.glob(f'{results_dir}/*.cif'))
converted = []
for cif in cif_files:
    name = os.path.basename(cif).replace('.cif', '')
    pdb  = os.path.join(pdb_dir, name + '.pdb')
    try:
        cif_to_pdb(cif, pdb)
        converted.append({'name': name, 'cif': cif, 'pdb': pdb})
    except Exception as e:
        print(f'  SKIP {name}: {e}')

print(f'Converted {len(converted)} CIF → PDB')
for c in converted[:5]:
    print(f"  {c['name']}")

In [ ]:
#@title 🔬 Load ProteinMPNN
import copy, torch, numpy as np
sys.path.append('/content/ProteinMPNN')
from protein_mpnn_utils import (
    loss_nll, _scores, _S_to_seq, tied_featurize, parse_PDB,
    StructureDatasetPDB, ProteinMPNN
)

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')

checkpoint_path = '/content/ProteinMPNN/vanilla_model_weights/v_48_020.pt'
checkpoint = torch.load(checkpoint_path, map_location=device)

hidden_dim = 128; num_layers = 3
mpnn_model = ProteinMPNN(
    num_letters=21, node_features=hidden_dim, edge_features=hidden_dim,
    hidden_dim=hidden_dim, num_encoder_layers=num_layers,
    num_decoder_layers=num_layers, augment_eps=0.0,
    k_neighbors=checkpoint['num_edges']
)
mpnn_model.to(device)
mpnn_model.load_state_dict(checkpoint['model_state_dict'])
mpnn_model.eval()
print(f'ProteinMPNN loaded  |  edges={checkpoint["num_edges"]}  |  noise={checkpoint["noise_level"]}A')

In [ ]:
#@title 🧪 Run ProteinMPNN — redesign binder (chain A), fix RBX1 (chain B)
import csv

MPNN_TEMPS     = [0.05, 0.10, 0.15]  # conservative → diverse
SEQS_PER_TEMP  = 3
alphabet       = 'ACDEFGHIKLMNPQRSTVWYX'
omit_AAs_np    = np.array([aa in 'X' for aa in alphabet], dtype=np.float32)
bias_AAs_np    = np.zeros(len(alphabet))

# Merge with BoltzDesign1 scores for metadata
bd_scores = {}
if os.path.exists(results_csv):
    for _, row in pd.read_csv(results_csv).iterrows():
        bd_scores[str(row.get('target',''))] = {
            'iptm': row.get('iptm', ''),
            'complex_plddt': row.get('complex_plddt', ''),
            'length': row.get('length', ''),
        }

all_seqs = []  # {name, seq, backbone, temp, bd_iptm}

for entry in converted:
    pdb_path = entry['pdb']
    backbone = entry['name']

    # Parse PDB — design chain A, fix chain B
    try:
        pdb_dict_list = parse_PDB(pdb_path, input_chain_list=['A', 'B'])
    except Exception as e:
        print(f'  SKIP {backbone} (parse error): {e}'); continue

    if not pdb_dict_list: continue

    dataset = StructureDatasetPDB(pdb_dict_list, truncate=None, max_length=20000)
    chain_id_dict = {pdb_dict_list[0]['name']: (['A'], ['B'])}  # design A, fix B

    for temp in MPNN_TEMPS:
        with torch.no_grad():
            for protein in dataset:
                batch = [copy.deepcopy(protein) for _ in range(SEQS_PER_TEMP)]
                try:
                    X, S, mask, lengths, chain_M, chain_encoding_all, \
                    chain_list_list, visible_list_list, masked_list_list, \
                    masked_chain_length_list_list, chain_M_pos, omit_AA_mask, \
                    residue_idx, dihedral_mask, tied_pos_list_of_lists_list, \
                    pssm_coef, pssm_bias, pssm_log_odds_all, bias_by_res_all, \
                    tied_beta = tied_featurize(
                        batch, device, chain_id_dict,
                        None, None, None, None, None
                    )
                except Exception as e:
                    print(f'  SKIP {backbone} T={temp}: {e}'); continue

                sample_dict = mpnn_model.sample(
                    X, torch.randn(chain_M.shape, device=device),
                    S, chain_M, chain_encoding_all, residue_idx,
                    mask=mask, temperature=temp,
                    omit_AAs_np=omit_AAs_np, bias_AAs_np=bias_AAs_np,
                    chain_M_pos=chain_M_pos, omit_AA_mask=omit_AA_mask,
                    pssm_coef=pssm_coef, pssm_bias=pssm_bias,
                    pssm_multi=0.0, pssm_log_odds_flag=False,
                    pssm_log_odds_mask=(pssm_log_odds_all > 0.0).float(),
                    pssm_bias_flag=False, bias_by_res=bias_by_res_all
                )
                S_sample = sample_dict['S']  # (batch, L)

                for b_ix in range(SEQS_PER_TEMP):
                    # Extract only chain A (binder) residues
                    full_seq = _S_to_seq(S_sample[b_ix], chain_M[b_ix])
                    # chain A is the designed portion — get its length from pdb_dict
                    binder_seq = pdb_dict_list[0].get('seq_chain_A', '')
                    binder_len = len(binder_seq)
                    # full_seq = binder + target concatenated; take first binder_len chars
                    designed_seq = full_seq[:binder_len]

                    name = f'BZD_{backbone}_t{temp:.2f}_s{b_ix+1}'
                    bd_meta = bd_scores.get(backbone, {})
                    all_seqs.append({
                        'name': name,
                        'seq': designed_seq,
                        'length': len(designed_seq),
                        'backbone': backbone,
                        'temp': temp,
                        'bd_iptm': bd_meta.get('iptm', ''),
                        'bd_plddt': bd_meta.get('complex_plddt', ''),
                    })

print(f'\nGenerated {len(all_seqs)} sequences')
print(f'  {len(converted)} backbones × {len(MPNN_TEMPS)} temps × {SEQS_PER_TEMP} seqs = {len(converted)*len(MPNN_TEMPS)*SEQS_PER_TEMP} expected')
print(f'\nSample:')
for s in all_seqs[:6]:
    print(f"  {s['name']:<50} {s['length']} AA  bd_ipTM={s['bd_iptm']}")

In [ ]:
#@title 💾 Write FASTA + Boltz-2 YAMLs

# RBX1 RING core sequence (res 32-83, 52 AA) — truncated, no disordered tails
RBX1_RING_CORE = 'LWAWDIVVDNCAICRNHIMDLCIECQANQASATSEECTVAWGVCNHAFHFHC'

OUT_DIR  = 'bzd_mpnn_outputs'
YAML_DIR = f'{OUT_DIR}/boltz_yamls'
os.makedirs(YAML_DIR, exist_ok=True)

# Sort: best BoltzDesign1 backbones first, then by temp
all_seqs.sort(key=lambda s: (-(float(s['bd_iptm']) if s['bd_iptm'] else 0), s['temp']))

# FASTA
fasta_path = f'{OUT_DIR}/bzd_mpnn_candidates.fasta'
with open(fasta_path, 'w') as f:
    for s in all_seqs:
        f.write(f">{s['name']}\n{s['seq']}\n")
print(f'FASTA: {fasta_path}  ({len(all_seqs)} sequences)')

# Boltz-2 YAMLs (using truncated RBX1)
yaml_tmpl = 'version: 1\nsequences:\n  - protein:\n      id: A\n      sequence: {binder}\n  - protein:\n      id: B\n      sequence: {rbx1}\n'
for s in all_seqs:
    with open(f"{YAML_DIR}/{s['name']}_complex.yaml", 'w') as f:
        f.write(yaml_tmpl.format(binder=s['seq'], rbx1=RBX1_RING_CORE))
print(f'Boltz-2 YAMLs: {YAML_DIR}/  ({len(all_seqs)} files)')

# Metadata CSV
csv_path = f'{OUT_DIR}/bzd_mpnn_candidates.csv'
with open(csv_path, 'w', newline='') as f:
    import csv as _csv
    w = _csv.DictWriter(f, fieldnames=['name','backbone','temp','length','bd_iptm','bd_plddt','seq'])
    w.writeheader(); w.writerows(all_seqs)
print(f'CSV: {csv_path}')

In [ ]:
#@title 🔬 Rescore with Boltz-2 (install if needed)
try:
    import boltz as _b
    print('Boltz-2 available')
except ImportError:
    print('Installing Boltz-2...')
    !pip install -q boltz

import glob, json, subprocess

BOLTZ_OUT = f'{OUT_DIR}/boltz_results'
os.makedirs(BOLTZ_OUT, exist_ok=True)

yaml_files = sorted(glob.glob(f'{YAML_DIR}/*.yaml'))
print(f'Scoring {len(yaml_files)} sequences with Boltz-2...')

for yf in yaml_files:
    name = os.path.basename(yf).replace('_complex.yaml', '')
    out  = f'{BOLTZ_OUT}/{name}'
    if os.path.exists(f'{out}/predictions'):
        continue  # already scored
    !boltz predict {yf} --out_dir {out} --accelerator gpu \
        --recycling_steps 3 --sampling_steps 200 --diffusion_samples 1 \
        --num_workers 2 2>/dev/null

print('Scoring complete')

In [ ]:
#@title 🏆 Filter and rank results
import json, glob

scored = []
for s in all_seqs:
    conf_pattern = f"{BOLTZ_OUT}/{s['name']}/predictions/*/confidence_*.json"
    conf_files = glob.glob(conf_pattern)
    if not conf_files:
        continue
    with open(conf_files[0]) as f:
        conf = json.load(f)
    iptm   = conf.get('iptm', 0)
    iplddt = conf.get('interface_plddt', conf.get('complex_plddt', 0))
    scored.append({**s, 'iptm': iptm, 'iplddt': iplddt})

scored.sort(key=lambda r: -r['iptm'])
passing    = [r for r in scored if r['iptm'] >= 0.70]
borderline = [r for r in scored if 0.65 <= r['iptm'] < 0.70]

print(f'Scored:       {len(scored)}')
print(f'PASS ≥0.70:   {len(passing)}')
print(f'BORDER ≥0.65: {len(borderline)}')
print(f'FAIL <0.65:   {len(scored)-len(passing)-len(borderline)}')

print(f'\nTop 20:')
print(f'{"name":<50} {"ipTM":>6} {"ipLDDT":>7} {"bd_ipTM":>8} {"len":>5} {"T":>5}')
print('-'*80)
for r in scored[:20]:
    print(f"{r['name']:<50} {r['iptm']:>6.3f} {r['iplddt']:>7.3f} "
          f"{float(r['bd_iptm']) if r['bd_iptm'] else 0:>8.3f} "
          f"{r['length']:>5} {r['temp']:>5}")

In [ ]:
#@title 📦 Save and download passing sequences
from google.colab import files
import csv as _csv

PASS_FASTA = f'{OUT_DIR}/bzd_passing.fasta'
PASS_CSV   = f'{OUT_DIR}/bzd_passing.csv'

with open(PASS_FASTA, 'w') as f:
    for r in passing + borderline:
        f.write(f">{r['name']} iptm={r['iptm']:.3f} bd_iptm={r['bd_iptm']}\n{r['seq']}\n")

with open(PASS_CSV, 'w', newline='') as f:
    w = _csv.DictWriter(f, fieldnames=['name','backbone','temp','length','iptm','iplddt','bd_iptm','bd_plddt','seq'])
    w.writeheader(); w.writerows(passing + borderline)

print(f'Saved {len(passing+borderline)} sequences (PASS + BORDERLINE)')
files.download(PASS_FASTA)
files.download(PASS_CSV)